# {Project Title}📝

![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
📝 <!-- Answer Below -->

The purchasing of a home is a complicated process. With this project, I want to make the process simpler—or at least enumerate all one's options better so that an informed decision can be made by the buyer. The Cincinnati housing market is complex and this project aims to summarize the market and infer home pricing validity.

## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
📝 <!-- Answer Below -->

The broad question this project aims to answer is: what does the Cincinnati housing market look like at present?

I plan to break this down into the following questions:
What is the average home price in Cincinnati? The median?
What is the distribution of home prices? Distribution of square footage?
How do home prices and square footage correlate, if at all?
Given a house's square footage and other factors, can I predict what a fair home price would be?
Which areas in Cincinnati are safest for buying homes?

## What would an answer look like?
*What is your hypothesized answer to your question?*
📝 <!-- Answer Below -->

I want to use box plots to demonstrate the distribution of housing prices and size of house. Bar graphs for price by neighborhood. Scatter plot for home price and square footage. I want to use machine learning to define an appropriate house price based on parameters; I think classification would work if I assigned a range of prices, or maybe regression for more accuracy. A map displaying crime reports or a bar chart with crimes by neighborhood would also be interesting. With these visualizations, I should be able to draw some useful information from them.

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*
📝 <!-- Answer Below -->

I plan to use primarily a dataset from the Rentcast API (/listings/sale endpoint), which gives a lot of the same information as MLS systems. I will limit the API call to just Cincinnati and loop through to get all the offerings in the area.
I also plan to use the Cincinnati CPD Crime Reports dataset, which I will download directly from the website and put in my assets folder.
To be able to compare these, I want to utilize neighborhood as an indicator for crime level against housing location. The Rentcast API doesn't have the neighborhood (i.e. Avondale, Over the Rhine, Madisonville, etc.) so I will need to bring in an additional API for that. I will run each Rentcast listing address against the Cincinnati provided ArcGIS API service and plot each lat/long for the house against the geojson provided by that API. This will give me neighborhoods within the city limits, so Fairfax for instance would not be provided. However, I should still get a good number of usable results, and the crime dataset also has the same limitations about location.

Please also note that the Rentcast API is a paid API, so I need to be careful about how much I use it. I will include my request code in this document but will save the data to a CSV so that I only have to call the API one time. This will also make it easier for me to run each address against the ArcGIS API.

Overall, the model would essentially be as such: [Rentcast listings --M:1-- Geocoding neighborhood --1:M-- Crime reports] so the ArcGIS API acts as a cross-reference table between the listings and the crimes.

## Approach and Analysis
*What is your approach to answering your project question?*
*How will you use the identified data to answer your project question?*
📝 <!-- Start Discussing the project here; you can add as many code cells as you need -->

I plan to use a variety of analysis techniques in the development of this project. I will primarily use quantitative techniques to summarize and explain the current Cincinnati housing market. I will also develop predictive models for identifying future trends. The Rentcast API, Cincinnati crime reports dataset, and Cincinnati ArcGIS API mesh well together to be able to create a descriptive picture of the housing market. There is good quantitative data already stored in the datasets (like price or square footage) that can be analyzed further.

In [1]:
# importing my libraries
import pandas as pd
import requests
import os
from dotenv import load_dotenv, find_dotenv
import geopandas as gpd

In [19]:
# API url for rentcast
rentcastURL = "https://api.rentcast.io/v1/listings/sale?city=Cincinnati&state=OH&limit=500&includeTotalCount=true"
load_dotenv(find_dotenv())
headers = {
    "X-Api-Key": os.getenv("rentcast-api")
}
offset = 0
null = None
lastPage = False

# looping through each page of the API and appending it to a CSV
while lastPage == False:
    # query the api
    response = requests.get(rentcastURL + f"&offset={offset}", headers=headers)
    df = response.json()

    # include a mapping of fields since the API response seems to change across offsets, messing up the csv
    field_mapping = {
        "address": "formattedAddress",
        "addressLine1": "addressLine1",
        "addressLine2": "addressLine2",
        "city": "city",
        "state": "state",
        "zipCode": "zipCode",
        "county": "county",
        "latitude": "latitude",
        "longitude": "longitude",
        "propertyType": "propertyType",
        "bedrooms": "bedrooms",
        "bathrooms": "bathrooms",
        "squareFootage": "squareFootage",
        "lotSize": "lotSize",
        "yearBuilt": "yearBuilt",
        "status": "status",
        "price": "price",
        "listingType": "listingType",
        "listedDate": "listedDate",
        "daysOnMarket": "daysOnMarket"
    }

    df = pd.DataFrame([
        {
            df_column: property_data.get(json_key)
            for df_column, json_key in field_mapping.items()
        }
        for property_data in df
    ])

    # include headers for the first response and not for the subsequent responses
    if offset != 0:
        df.to_csv("assets/RentCastAPI.csv", mode='a', header=False, index=False, sep=",")
    else:
        df.to_csv("assets/RentCastAPI.csv", header=True, index=False, sep=",")

    # increase the offset by the limit (500) to go to the next page
    offset += 500

    # if offset larger than the total count of listings, end the loop
    if offset >= int(response.headers['x-total-count']):
        lastPage = True
    

In [ ]:
# getting the geojson for each neighborhood in Cincinnati
# this is my alternative to the Geocoding API I was originally going to use because that would take way too long to run to be feasible
# using the Cincinnati ArcGIS service
url = (
    "https://services.arcgis.com/JyZag7oO4NteHGiq/"
    "ArcGIS/rest/services/Open_Data_Feature_Collection/"
    "FeatureServer/12/query"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "geojson"
}

response = requests.get(url, params=params)
response.raise_for_status()

# write the geojson to a file
with open("assets/cincinnatiNeighborhoods.geojson", "wb") as f:
    f.write(response.content)

In [20]:
# load neighborhoods and listings
neighborhoods = gpd.read_file("assets/cincinnatiNeighborhoods.geojson")
listings = pd.read_csv("assets/RentCastAPI.csv")

# setting the lat/long for each listing in the appropriate coordinate system
geoListings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(
        listings["longitude"],
        listings["latitude"]
    ),
    crs="EPSG:4326"
)

neighborhoods = neighborhoods.to_crs(geoListings.crs)

# join together the neighborhoods and the listings
matched = gpd.sjoin(
    geoListings,
    neighborhoods[["SNA_NAME", "geometry"]],
    how="left",
    predicate="within"
)
matched = matched.rename(columns={'SNA_NAME': 'neighborhood'})

# output the listings with neighborhoods to a csv
matched.to_csv("assets/AllListings.csv")

C:\Users\sageb\AppData\Local\Temp\ipykernel_76352\210221624.py:3: DtypeWarning: Columns (17: listingType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("assets/RentCastAPI.csv")


In [21]:
# from here, getting all my data in one spot for future analysis and use
listingdf = pd.read_csv("assets/AllListings.csv")
crimedf = pd.read_csv("assets/CincinnatiCrimeReports.csv")

C:\Users\sageb\AppData\Local\Temp\ipykernel_76352\1958847247.py:2: DtypeWarning: Columns (18: listingType) have mixed types. Specify dtype option on import or set low_memory=False.
  listingdf = pd.read_csv("assets/AllListings.csv")


EXPLORATORY DATA ANALYSIS

In [22]:
listingdf.sample(3)

,Unnamed: 0,address,addressLine1,addressLine2,city,state,zipCode,county,latitude,longitude,...,lotSize,yearBuilt,status,price,listingType,listedDate,daysOnMarket,geometry,index_right,neighborhood
17194,17194,"381 Rosemont Ave, Cincinnati, OH 45204",381 Rosemont Ave,NaN,Cincinnati,OH,45204,Hamilton,39.099041,-84.580046,...,5924.0,1950.0,Inactive,79900,NaN,2023-06-07T00:00:00.000Z,251,POINT (-84.580046 39.099041),36.0,West Price Hill
18708,18708,"1825 Emerson Ave, Cincinnati, OH 45239",1825 Emerson Ave,NaN,Cincinnati,OH,45239,Hamilton,39.211347,-84.556055,...,4792.0,1931.0,Inactive,179900,NaN,2023-11-25T00:00:00.000Z,5,POINT (-84.556055 39.211347),NaN,NaN
34645,34645,"724 Main St, Unit 6, Cincinnati, OH 45202",724 Main St,Unit 6,Cincinnati,OH,45202,Hamilton,39.104703,-84.510184,...,NaN,NaN,Inactive,179900,NaN,2020-09-21T20:58:20.903Z,685,POINT (-84.510184 39.104703),17.0,Downtown


In [7]:
listingdf.info()
listingdf.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 33652 entries, 0 to 33651
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             33652 non-null  int64  
 1   id                     33652 non-null  str    
 2   formattedAddress       33652 non-null  str    
 3   addressLine1           33652 non-null  str    
 4   addressLine2           3543 non-null   str    
 5   city                   33652 non-null  str    
 6   state                  33652 non-null  str    
 7   stateFips              33652 non-null  int64  
 8   zipCode                33652 non-null  int64  
 9   county                 33652 non-null  str    
 10  countyFips             33641 non-null  float64
 11  latitude               33652 non-null  float64
 12  longitude              33652 non-null  float64
 13  propertyType           33649 non-null  str    
 14  lotSize                29893 non-null  float64
 15  status       

Unnamed: 0                   0
id                           0
formattedAddress             0
addressLine1                 0
addressLine2             30109
city                         0
state                        0
stateFips                    0
zipCode                      0
county                       0
countyFips                  11
latitude                     0
longitude                    0
propertyType                 3
lotSize                   3759
status                    4433
price                     3966
listingType               2231
listedDate                1760
removedDate                500
createdDate                  0
lastSeenDate               487
daysOnMarket               144
mlsName                    709
mlsNumber                 9155
listingAgent.name         9377
listingAgent.phone       20247
listingAgent.email       20840
listingOffice.name       21883
listingOffice.phone      22574
listingOffice.email      21233
listingOffice.website    21809
listingA

In [14]:
listingdf['price'] = pd.to_numeric(listingdf['price'], errors='coerce')
listingdf['daysOnMarket'] = pd.to_numeric(listingdf['daysOnMarket'], errors='coerce')
listingdf['bedrooms'] = pd.to_numeric(listingdf['bedrooms'], errors='coerce')
listingdf['bathrooms'] = pd.to_numeric(listingdf['bathrooms'], errors='coerce')
listingdf['yearBuilt'] = pd.to_numeric(listingdf['yearBuilt'], errors='coerce')
listingdf['listedDate'] = pd.to_datetime(listingdf['listedDate'])
listingdf['removedDate'] = pd.to_datetime(listingdf['removedDate'])
listingdf['createdDate'] = pd.to_datetime(listingdf['createdDate'])

ValueError: time data "1890.0" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [12]:
listingdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 33652 entries, 0 to 33651
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             33652 non-null  int64  
 1   id                     33652 non-null  str    
 2   formattedAddress       33652 non-null  str    
 3   addressLine1           33652 non-null  str    
 4   addressLine2           3543 non-null   str    
 5   city                   33652 non-null  str    
 6   state                  33652 non-null  str    
 7   stateFips              33652 non-null  int64  
 8   zipCode                33652 non-null  int64  
 9   county                 33652 non-null  str    
 10  countyFips             33641 non-null  float64
 11  latitude               33652 non-null  float64
 12  longitude              33652 non-null  float64
 13  propertyType           33649 non-null  str    
 14  lotSize                29893 non-null  float64
 15  status       

In [13]:
listingdf.head()

,Unnamed: 0,id,formattedAddress,addressLine1,addressLine2,city,state,stateFips,zipCode,county,...,listingOffice.website,listingAgent.website,bedrooms,bathrooms,yearBuilt,hoa.fee,squareFootage,geometry,index_right,neighborhood
0,0,"2342-Banning-Rd,-Cincinnati,-OH-45239","2342 Banning Rd, Cincinnati, OH 45239",2342 Banning Rd,NaN,Cincinnati,OH,39,45239,Hamilton,...,www.dwyercre.com,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-84.568619 39.20697),NaN,NaN
1,1,"2352-Iroll-Ave,-Cincinnati,-OH-45225","2352 Iroll Ave, Cincinnati, OH 45225",2352 Iroll Ave,NaN,Cincinnati,OH,39,45225,Hamilton,...,https://westernhills.huff.com/,https://www.coldwellbankerhomes.com/oh/cincinn...,NaN,NaN,NaN,NaN,NaN,POINT (-84.562771 39.151662),24.0,East Westwood
2,2,"720-Main-St,-Unit-4,-Cincinnati,-OH-45202","720 Main St, Unit 4, Cincinnati, OH 45202",720 Main St,Unit 4,Cincinnati,OH,39,45202,Hamilton,...,http://www.comey.com/,NaN,2.0,1.0,1884.0,288.0,NaN,POINT (-84.510184 39.104704),17.0,Downtown
3,3,"1226-Rutledge-Ave,-Cincinnati,-OH-45205","1226 Rutledge Ave, Cincinnati, OH 45205",1226 Rutledge Ave,NaN,Cincinnati,OH,39,45205,Hamilton,...,NaN,https://3cre.com/tryfon-christoforou/,0.0,NaN,1925.0,NaN,NaN,POINT (-84.58735 39.11435),36.0,West Price Hill
4,4,"1938-Bigelow-St,-Unit-2,-Cincinnati,-OH-45219","1938 Bigelow St, Unit 2, Cincinnati, OH 45219",1938 Bigelow St,Unit 2,Cincinnati,OH,39,45219,Hamilton,...,www.sibcycline.com,https://agent.sibcycline.com/jback,3.0,3.5,NaN,198.0,2222.0,POINT (-84.508041 39.117386),21.0,Mt. Auburn


## Resources and References
*What resources and references have you used for this project?*
📝 <!-- Answer Below -->

https://services.arcgis.com/JyZag7oO4NteHGiq/ArcGIS/rest/services/Open_Data_Feature_Collection/FeatureServer/12

https://developers.rentcast.io/reference/introduction

In [18]:
# ⚠️ Make sure you run this cell at the end of your notebook before every submission!
!jupyter nbconvert --to python source.ipynb

[NbConvertApp] Converting notebook source.ipynb to python
[NbConvertApp] Writing 8055 bytes to source.py
